In [1]:
# Install laspy with laszip backend to parse compressed .laz files
!pip install -q laspy[laszip] numpy torch

from google.colab import drive
import laspy
import numpy as np
import torch

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def dilated_sampling_and_grouping(xyz, features, n_samples, k, dilation_ratio=1.2):
    """
    Fast dilated sampling and grouping:
    1. Randomly sample sigma * N_s centroids.
    2. Group k neighbors via KNN.
    3. Calculate average distance to neighbors (sparsity indicator).
    4. Pick top N_s centroids with the largest distances to prevent clustering in dense zones.
    """
    B, N, _ = xyz.shape
    n_dilated = int(n_samples * dilation_ratio)

    # 1. Random centroid sampling
    rand_idx = torch.randperm(N, device=xyz.device)[:n_dilated]
    sampled_xyz = xyz[:, rand_idx, :]
    sampled_features = features[:, rand_idx, :]

    # 2. Group neighbors around sampled centroids
    dist = torch.cdist(sampled_xyz, xyz)  # (B, n_dilated, N)
    _, knn_idx = torch.topk(dist, k=k, dim=-1, largest=False)  # (B, n_dilated, k)

    batch_idx = torch.arange(B, device=xyz.device).view(B, 1, 1).expand(-1, n_dilated, k)
    grouped_xyz = xyz[batch_idx, knn_idx]  # (B, n_dilated, k, 3)
    grouped_feat = features[batch_idx, knn_idx]  # (B, n_dilated, k, C)

    # 3. Compute mean distance between centroid and its k neighbors
    mean_dist = torch.norm(grouped_xyz - sampled_xyz.unsqueeze(2), dim=-1).mean(dim=-1)  # (B, n_dilated)

    # 4. Select top N_s most spread-out (sparse) regions
    _, top_idx = torch.topk(mean_dist, k=n_samples, dim=-1, largest=True)  # (B, n_samples)
    top_batch = torch.arange(B, device=xyz.device).view(B, 1).expand(-1, n_samples)

    final_xyz = sampled_xyz[top_batch, top_idx]  # (B, n_samples, 3)
    final_grouped_feat = grouped_feat[top_batch, top_idx]  # (B, n_samples, k, C)

    return final_xyz, final_grouped_feat


class ResidualBlock2D(nn.Module):
    """Local 2D Residual Block applied over (B, C, N_s, k)."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = F.relu(out + residual)
        return out


class ResidualBlock1D(nn.Module):
    """Global 1D Residual Block applied over (B, C, N_s)."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm1d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = F.relu(out + residual)
        return out


class GroupedSelfAttention(nn.Module):
    """Lightweight Attentional Weighting Module with group-wise 1x1 convolutions."""
    def __init__(self, in_dim, groups):
        super().__init__()
        self.groups = groups
        self.in_dim = in_dim
        self.q_conv = nn.Conv1d(in_dim, in_dim, kernel_size=1, groups=groups, bias=False)
        self.k_conv = nn.Conv1d(in_dim, in_dim, kernel_size=1, groups=groups, bias=False)
        self.v_conv = nn.Conv1d(in_dim, in_dim, kernel_size=1, bias=False)

    def forward(self, x):
        # x: (B, C, N_s)
        B, C, N_s = x.shape
        Q = self.q_conv(x).view(B, self.groups, C // self.groups, N_s)
        K = self.k_conv(x).view(B, self.groups, C // self.groups, N_s)
        V = self.v_conv(x)  # (B, C, N_s)

        # Summed attention matrix across groups: (B, N_s, N_s)
        W = torch.einsum('bgcn,bgcm->bnm', Q, K)
        attention_map = F.softmax(W / (C ** 0.5), dim=-1)

        # Apply attention map to value features
        f_atten = torch.bmm(V, attention_map.transpose(-1, -2)) + x
        return f_atten


class GeMPooling(nn.Module):
    """Generalized Mean (GeM) Pooling Layer."""
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p_init)
        self.eps = eps

    def forward(self, x):
        # x: (B, C, N_s)
        x_clamped = x.clamp(min=self.eps)
        gem = (torch.mean(x_clamped ** self.p, dim=-1)) ** (1.0 / self.p)
        return gem  # (B, C)


class FeatureEncodingLayer(nn.Module):
    """Hierarchical Encoding Layer: Dilated Grouping -> ResMLP 2D -> MaxPool -> Grouped SA -> ResMLP 1D."""
    def __init__(self, n_samples, k, in_dim, out_dim, groups, dilation_ratio=1.2):
        super().__init__()
        self.n_samples = n_samples
        self.k = k
        self.dilation_ratio = dilation_ratio

        # MLP dimension mapping: [C_in -> C_out]
        self.mlp = nn.Sequential(
            nn.Conv2d(in_dim, out_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_dim),
            nn.ReLU(inplace=True)
        )
        self.local_res = ResidualBlock2D(out_dim)
        self.attention = GroupedSelfAttention(out_dim, groups)
        self.global_res = ResidualBlock1D(out_dim)

    def forward(self, xyz, features):
        # Dilated sampling & neighborhood grouping
        sampled_xyz, grouped_features = dilated_sampling_and_grouping(
            xyz, features, self.n_samples, self.k, self.dilation_ratio
        )

        # Shape to (B, C_in, N_s, k)
        grouped_features = grouped_features.permute(0, 3, 1, 2).contiguous()

        # Local residual encoding & channel expansion
        feat = self.mlp(grouped_features)
        feat = self.local_res(feat)

        # Regional aggregation (Max-pooling over k)
        feat_agg = torch.max(feat, dim=-1)[0]  # (B, C_out, N_s)

        # Attentional weighting & global residual encoding
        feat_atten = self.attention(feat_agg)
        feat_global = self.global_res(feat_atten)

        # Output features shaped for next layer: (B, N_s, C_out)
        out_features = feat_global.transpose(1, 2).contiguous()
        return sampled_xyz, out_features, feat_global


class LWRNet(nn.Module):
    """LWR-Net Architecture."""
    def __init__(self):
        super().__init__()
        # Layer 1: N_s = 1024, k = 32, C: 3 -> 64, G = 2
        self.layer1 = FeatureEncodingLayer(
            n_samples=1024, k=32, in_dim=3, out_dim=64, groups=2, dilation_ratio=1.2
        )
        # Layer 2: N_s = 256, k = 16, C: 64 -> 256, G = 4
        self.layer2 = FeatureEncodingLayer(
            n_samples=256, k=16, in_dim=64, out_dim=256, groups=4, dilation_ratio=1.2
        )
        # Feature fusion: GeM pooling to 256-D global descriptor
        self.gem_pool = GeMPooling(p_init=3.0)

    def forward(self, x):
        """
        Input:
            x: (B, N, 3) raw 3D coordinates
        Returns:
            global_descriptor: (B, 256) L2-normalized descriptor
        """
        # Layer 1
        xyz_l1, feat_l1, _ = self.layer1(x, x)

        # Layer 2
        _, _, feat_l2_tensor = self.layer2(xyz_l1, feat_l1)

        # GeM aggregation: (B, 256, 256) -> (B, 256)
        global_descriptor = self.gem_pool(feat_l2_tensor)

        # L2-normalization for cosine/Euclidean retrieval
        global_descriptor = F.normalize(global_descriptor, p=2, dim=-1)
        return global_descriptor

In [3]:
import laspy
import numpy as np
import torch
from pyproj import Transformer


def load_preprocess_and_transform_laz(
    laz_path,
    src_crs="EPSG:6348",   # NAD83 (2011) / UTM Zone 19N (Meters)
    tgt_crs="EPSG:32619",  # WGS 84 / UTM Zone 19N (nuScenes Boston standard)
    target_points=4096,
    ground_percentile=5
):
    """
    Loads a .laz file, extracts and reprojects its global centroid,
    and returns both the normalized tensor for LWR-Net and the transformed coordinates.
    """
    # 1. Read .laz point coordinates
    las = laspy.read(laz_path)
    points = np.vstack((las.x, las.y, las.z)).transpose().astype(np.float32)

    # 2. Extract global centroid and transform to target CRS
    raw_centroid = np.mean(points, axis=0)
    transformer = Transformer.from_crs(src_crs, tgt_crs, always_xy=True)
    trans_x, trans_y = transformer.transform(raw_centroid[0], raw_centroid[1])
    global_coord = np.array([trans_x, trans_y])

    # 3. Ground filtering
    if hasattr(las, "classification") and np.any(las.classification == 2):
        points = points[las.classification != 2]
    else:
        z_cutoff = np.percentile(points[:, 2], ground_percentile)
        points = points[points[:, 2] > z_cutoff]

    # 4. Uniform downsampling to exactly 4096 points
    num_pts = points.shape[0]
    if num_pts >= target_points:
        idx = np.random.choice(num_pts, target_points, replace=False)
    else:
        idx = np.random.choice(num_pts, target_points, replace=True)
    points = points[idx]

    # 5. Shift to zero mean and rescale to [-1, 1] range
    points -= np.mean(points, axis=0, keepdims=True)
    max_bound = np.max(np.abs(points))
    if max_bound > 0:
        points /= max_bound

    tensor_input = torch.from_numpy(points).unsqueeze(0).float()
    return tensor_input, global_coord


# --- Execution Block ---
drive_path = '/content/drive/MyDrive/AH2179/ma2021_cent_east_J1448920/ma2021_cent_east_J1448920tR0_C0.laz'

# Preprocess point cloud and retrieve aligned global coordinate
submap_tensor, submap_global_xy = load_preprocess_and_transform_laz(
    drive_path,
    src_crs="EPSG:6348",   # NAD83 2011 UTM 19N
    tgt_crs="EPSG:32619"   # WGS 84 UTM 19N
)

print(f"Preprocessed Tensor Shape: {submap_tensor.shape}")  # torch.Size([1, 4096, 3])
print(f"Aligned Global Coordinate (UTM 19N): X={submap_global_xy[0]:.2f}m, Y={submap_global_xy[1]:.2f}m")

# Inference test with LWR-Net
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LWRNet().to(device)
model.eval()

with torch.no_grad():
    descriptor = model(submap_tensor.to(device))

print(f"Output Descriptor Dimension: {descriptor.shape}")  # torch.Size([1, 256])

Preprocessed Tensor Shape: torch.Size([1, 4096, 3])
Aligned Global Coordinate (UTM 19N): X=780424.06m, Y=2949421.75m
Output Descriptor Dimension: torch.Size([1, 256])


In [4]:
import os
import json
import pandas as pd
import gc

# Path to the metadata folder containing ego_pose.json and sample_data.json
META_DIR = "/content/drive/MyDrive/AH2179/v1.0-trainval_meta/v1.0-trainval"

# Exact parent folder containing the 'samples/' directory discovered by glob
BLOBS_DIR = "/content/drive/MyDrive/AH2179"

print("Loading metadata JSON files...")
with open(os.path.join(META_DIR, "ego_pose.json"), "r") as f:
    ego_poses = {pose["token"]: pose["translation"] for pose in json.load(f)}

with open(os.path.join(META_DIR, "sample_data.json"), "r") as f:
    sample_data_list = json.load(f)

records = []
for entry in sample_data_list:
    # Match LiDAR keyframes
    if entry.get("is_key_frame") and "LIDAR_TOP" in entry.get("filename", ""):
        rel_path = entry["filename"]  # e.g., 'samples/LIDAR_TOP/n008-....pcd.bin'
        full_bin_path = os.path.join(BLOBS_DIR, rel_path)

        if os.path.exists(full_bin_path):
            trans = ego_poses[entry["ego_pose_token"]]
            records.append({
                "filepath": full_bin_path,
                "x_utm": trans[0],
                "y_utm": trans[1],
                "z_utm": trans[2],
                "timestamp": entry["timestamp"]
            })

df_scans = pd.DataFrame(records).sort_values("timestamp").reset_index(drop=True)
print(f"Successfully matched and sorted {len(df_scans)} scans with global coordinates.")
display(df_scans.head())

# Free up system RAM by deleting large dictionaries and lists
del sample_data_list
del ego_poses
del records
gc.collect()

Loading metadata JSON files...
Successfully matched and sorted 3376 scans with global coordinates.


,filepath,x_utm,y_utm,z_utm,timestamp
0,/content/drive/MyDrive/AH2179/samples/LIDAR_TO...,1010.132835,610.811165,0.0,1531883530449377
1,/content/drive/MyDrive/AH2179/samples/LIDAR_TO...,1010.261466,612.897202,0.0,1531883530949817
2,/content/drive/MyDrive/AH2179/samples/LIDAR_TO...,1010.029253,614.858429,0.0,1531883531450214
3,/content/drive/MyDrive/AH2179/samples/LIDAR_TO...,1009.423492,616.758643,0.0,1531883531950107
4,/content/drive/MyDrive/AH2179/samples/LIDAR_TO...,1008.359039,618.662797,0.0,1531883532449967


37

In [5]:
import numpy as np
import torch
import gc
from sklearn.neighbors import NearestNeighbors

def preprocess_nuscenes_scan(bin_path, target_points=4096, ground_percentile=5):
    # NuScenes binary points: [x, y, z, intensity, ring_index]
    scan = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 5)
    points = scan[:, :3]

    # Crop vehicle chassis returns
    dist_2d = np.sqrt(points[:, 0]**2 + points[:, 1]**2)
    points = points[dist_2d > 2.0]

    # Ground cutoff
    z_cutoff = np.percentile(points[:, 2], ground_percentile)
    points = points[points[:, 2] > z_cutoff]

    # Downsample uniformly to 4096 points
    n_pts = points.shape[0]
    if n_pts >= target_points:
        idx = np.random.choice(n_pts, target_points, replace=False)
    else:
        idx = np.random.choice(n_pts, target_points, replace=True)
    points = points[idx]

    # Centering to zero mean and [-1, 1] normalization
    points -= np.mean(points, axis=0, keepdims=True)
    max_bound = np.max(np.abs(points))
    if max_bound > 0:
        points /= max_bound

    return torch.from_numpy(points).float()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

descriptors_list = []
coords_list = []

# Take a random 5% subsample of scans
df_scans_subset = df_scans.sample(frac=0.05, random_state=42).reset_index(drop=True)

# Free up memory from the large original DataFrame
try:
    del df_scans
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Extracting 256-D descriptors for {len(df_scans_subset)} scans (5% subset)...")
with torch.no_grad():
    for _, row in df_scans_subset.iterrows():
        tensor_pts = preprocess_nuscenes_scan(row["filepath"]).unsqueeze(0).to(device)
        desc = model(tensor_pts)
        descriptors_list.append(desc.squeeze(0).cpu().numpy())
        coords_list.append([row["x_utm"], row["y_utm"]])

all_descriptors = np.vstack(descriptors_list)
all_coords = np.array(coords_list)

# Cleanup temporary lists before nearest neighbor search
del descriptors_list
del coords_list
gc.collect()

# Split sequence: Even indices as reference database, Odd indices as queries
db_idx = np.arange(0, len(all_descriptors), 2)
query_idx = np.arange(1, len(all_descriptors), 2)

db_desc, db_coords = all_descriptors[db_idx], all_coords[db_idx]
query_desc, query_coords = all_descriptors[query_idx], all_coords[query_idx]

# Query nearest neighbors in 256-D feature space
k_max = min(25, len(db_desc))
feature_nn = NearestNeighbors(n_neighbors=k_max, metric="euclidean")
feature_nn.fit(db_desc)
_, retrieved_indices = feature_nn.kneighbors(query_desc)

# Compute benchmark metrics
threshold_m = 25.0
num_queries = len(query_desc)
k_one_percent = max(1, int(round(len(db_desc) * 0.01)))

r_at_1, r_at_5, r_at_25, r_at_1pct = 0, 0, 0, 0
for i in range(num_queries):
    gt_pos = query_coords[i]
    top_matches = retrieved_indices[i]
    spatial_dist = np.linalg.norm(db_coords[top_matches] - gt_pos, axis=1)

    if spatial_dist[0] <= threshold_m:
        r_at_1 += 1
    if np.any(spatial_dist[:min(5, k_max)] <= threshold_m):
        r_at_5 += 1
    if np.any(spatial_dist[:k_max] <= threshold_m):
        r_at_25 += 1
    if np.any(spatial_dist[:k_one_percent] <= threshold_m):
        r_at_1pct += 1

print(f"\n--- Evaluation Results ({len(db_desc)} DB / {num_queries} Queries) ---")
print(f"Recall@1:                 {100.0 * r_at_1 / num_queries:.2f}%")
print(f"Recall@5:                 {100.0 * r_at_5 / num_queries:.2f}%")
print(f"Recall@25:                {100.0 * r_at_25 / num_queries:.2f}%")
print(f"Recall@1% (Top-{k_one_percent}):          {100.0 * r_at_1pct / num_queries:.2f}%")

Extracting 256-D descriptors for 169 scans (5% subset)...

--- Evaluation Results (85 DB / 84 Queries) ---
Recall@1:                 15.48%
Recall@5:                 23.81%
Recall@25:                36.90%
Recall@1% (Top-1):          15.48%
